# Proyecto 1 · Ingesta y diagnóstico inicial

**Curso:** CC3084 · Data Science  
**Alcance de este cuaderno:** unir los archivos crudos y diagnosticar su
estado inicial.



## 1. Dependencias y rutas

In [1]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAW_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORT_DIR = Path("../reports/diagnostico_inicial")

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Validación de la ingesta y unión de los 23 CSV

In [2]:
archivos = sorted(RAW_DIR.glob("*.csv"))

print("Archivos encontrados:", len(archivos))
display(pd.DataFrame({"archivo": [a.name for a in archivos]}))

if len(archivos) != 23:
    raise ValueError(
        f"Se esperaban 23 archivos y se encontraron {len(archivos)}."
    )

Archivos encontrados: 23


,archivo
0,ALTAVERAPAZ.csv
1,BAJAVERAPAZ.csv
2,CHIMALTENANGO.csv
3,CHIQUIMULA.csv
4,CIUDADCAPITAL.csv
5,ELPROGRESO.csv
6,ESCUINTLA.csv
7,GUATEMALA.csv
8,HUEHUETENANGO.csv
9,IZABAL.csv


In [3]:
dataframes = []
esquemas = []

for archivo in archivos:
    df_archivo = pd.read_csv(
        archivo,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    esquemas.append({
        "archivo": archivo.name,
        "registros": len(df_archivo),
        "variables": len(df_archivo.columns),
        "columnas": tuple(df_archivo.columns),
    })

    df_archivo["archivo_origen"] = archivo.name
    df_archivo["fila_origen"] = range(2, len(df_archivo) + 2)
    dataframes.append(df_archivo)

resumen_archivos = pd.DataFrame(esquemas)
display(resumen_archivos[["archivo", "registros", "variables"]])

columnas_referencia = esquemas[0]["columnas"]
resumen_archivos["esquema_consistente"] = (
    resumen_archivos["columnas"] == columnas_referencia
)

print(
    "Todos los archivos tienen el mismo esquema:",
    resumen_archivos["esquema_consistente"].all(),
)

,archivo,registros,variables
0,ALTAVERAPAZ.csv,476,17
1,BAJAVERAPAZ.csv,172,17
2,CHIMALTENANGO.csv,436,17
3,CHIQUIMULA.csv,240,17
4,CIUDADCAPITAL.csv,2162,17
5,ELPROGRESO.csv,159,17
6,ESCUINTLA.csv,709,17
7,GUATEMALA.csv,1909,17
8,HUEHUETENANGO.csv,592,17
9,IZABAL.csv,414,17


Todos los archivos tienen el mismo esquema: True


In [4]:
df_raw = pd.concat(dataframes, ignore_index=True)

ruta_unificada = (
    INTERIM_DIR
    / "establecimientos_diversificado_raw_unificado.csv"
)

df_raw.to_csv(
    ruta_unificada,
    index=False,
    encoding="utf-8-sig",
)

print("Dimensión del conjunto unido:", df_raw.shape)
print("Archivo creado:", ruta_unificada)
display(df_raw.head())

Dimensión del conjunto unido: (11890, 19)
Archivo creado: ..\data\interim\establecimientos_diversificado_raw_unificado.csv


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,archivo_origen,fila_origen
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ,ALTAVERAPAZ.csv,2
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTAVERAPAZ.csv,3
2,16-01-0139-46,16-01-0927,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,58015763,ARMANDO AUDILIO POP CUCUL,GILMA DOLORES GUAY PAZ DE LEAL,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTAVERAPAZ.csv,4
3,16-01-0140-46,16-01-0926,ALTA VERAPAZ,COBAN,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,78232301,JOSE ARTURO CHOC CHEN,VIRGINIA SOLANO SERRANO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTAVERAPAZ.csv,5
4,16-01-0141-46,16-01-0924,ALTA VERAPAZ,COBAN,ESCUELA NACIONAL DE CIENCIAS COMERCIALES,2A CALLE 11-10 ZONA 2,40389968,MOISÉS ADRIÁN LÓPEZ PÉREZ,ESTELA DEL CARMEN QUIM ROSALES,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ,ALTAVERAPAZ.csv,6


### Análisis de la ingesta

Escriba aquí la conclusión sobre:

- cantidad de archivos;
- consistencia de columnas;
- número total de registros;
- confirmación de que los archivos originales permanecen intactos.

## 3.a Número de registros y variables

In [5]:
columnas_originales = [
    columna for columna in df_raw.columns
    if columna not in ["archivo_origen", "fila_origen"]
]

filas_completamente_vacias = (
    df_raw[columnas_originales]
    .fillna("")
    .apply(lambda fila: fila.str.strip().eq("").all(), axis=1)
)

resumen_dimension = pd.DataFrame({
    "métrica": [
        "Registros crudos",
        "Registros no vacíos",
        "Filas completamente vacías",
        "Variables originales",
        "Columnas de trazabilidad",
    ],
    "valor": [
        len(df_raw),
        (~filas_completamente_vacias).sum(),
        filas_completamente_vacias.sum(),
        len(columnas_originales),
        2,
    ],
})

display(resumen_dimension)

,métrica,valor
0,Registros crudos,11890
1,Registros no vacíos,11867
2,Filas completamente vacías,23
3,Variables originales,17
4,Columnas de trazabilidad,2


### Análisis 3.a

> Escriba aquí su interpretación.

## 3.b Tipo de dato de cada variable

In [6]:
tipos = pd.DataFrame({
    "variable": df_raw.columns,
    "tipo_leído": df_raw.dtypes.astype(str).values,
})

display(tipos)

,variable,tipo_leído
0,CODIGO,string
1,DISTRITO,string
2,DEPARTAMENTO,string
3,MUNICIPIO,string
4,ESTABLECIMIENTO,string
5,DIRECCION,string
6,TELEFONO,string
7,SUPERVISOR,string
8,DIRECTOR,string
9,NIVEL,string


### Análisis 3.b

Recuerde que códigos, distritos y teléfonos deben conservarse como texto,
aunque contengan números.

## 3.c Cantidad y porcentaje de valores faltantes

In [7]:
marcadores_faltantes = {
    "", "N/A", "NA", "NULL", "-", ".", "SIN DATO",
    "S/D", "SD", "NO APLICA", "N.A."
}

faltantes = []

for columna in columnas_originales:
    normalizada = (
        df_raw[columna]
        .fillna("")
        .astype("string")
        .str.strip()
        .str.upper()
    )

    mascara = normalizada.isin(marcadores_faltantes)

    faltantes.append({
        "variable": columna,
        "faltantes": int(mascara.sum()),
        "porcentaje": round(mascara.mean() * 100, 4),
    })

tabla_faltantes = (
    pd.DataFrame(faltantes)
    .sort_values("faltantes", ascending=False)
)

display(tabla_faltantes)

,variable,faltantes,porcentaje
8,DIRECTOR,1814,15.2565
6,TELEFONO,969,8.1497
7,SUPERVISOR,558,4.6930
1,DISTRITO,555,4.6678
5,DIRECCION,104,0.8747
4,ESTABLECIMIENTO,28,0.2355
0,CODIGO,23,0.1934
3,MUNICIPIO,23,0.1934
2,DEPARTAMENTO,23,0.1934
9,NIVEL,23,0.1934


### Análisis 3.c

> Escriba aquí las variables más afectadas.

## 3.d Cantidad de valores únicos

In [8]:
tabla_unicos = pd.DataFrame({
    "variable": columnas_originales,
    "valores_unicos": [
        df_raw[columna].nunique(dropna=False)
        for columna in columnas_originales
    ],
}).sort_values("valores_unicos", ascending=False)

display(tabla_unicos)

,variable,valores_unicos
0,CODIGO,11868
5,DIRECCION,7527
4,ESTABLECIMIENTO,6668
6,TELEFONO,6572
8,DIRECTOR,5562
1,DISTRITO,1682
7,SUPERVISOR,1286
3,MUNICIPIO,353
16,DEPARTAMENTAL,27
2,DEPARTAMENTO,24


### Análisis 3.d

> Escriba aquí su interpretación.

## 3.e Registros duplicados exactos

In [9]:
duplicados_exactos = df_raw.duplicated(
    subset=columnas_originales,
    keep=False,
)

print(
    "Filas pertenecientes a grupos duplicados:",
    int(duplicados_exactos.sum()),
)

display(
    df_raw.loc[
        duplicados_exactos,
        columnas_originales + ["archivo_origen", "fila_origen"],
    ].sort_values(["CODIGO", "archivo_origen"])
)

Filas pertenecientes a grupos duplicados: 23


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL,archivo_origen,fila_origen
475,,,,,,,,,,,,,,,,,,ALTAVERAPAZ.csv,477
647,,,,,,,,,,,,,,,,,,BAJAVERAPAZ.csv,173
1083,,,,,,,,,,,,,,,,,,CHIMALTENANGO.csv,437
1323,,,,,,,,,,,,,,,,,,CHIQUIMULA.csv,241
3485,,,,,,,,,,,,,,,,,,CIUDADCAPITAL.csv,2163
3644,,,,,,,,,,,,,,,,,,ELPROGRESO.csv,160
4353,,,,,,,,,,,,,,,,,,ESCUINTLA.csv,710
6262,,,,,,,,,,,,,,,,,,GUATEMALA.csv,1910
6854,,,,,,,,,,,,,,,,,,HUEHUETENANGO.csv,593
7268,,,,,,,,,,,,,,,,,,IZABAL.csv,415


### Análisis 3.e

Verifique si los duplicados corresponden a registros reales o a filas
completamente vacías. No elimine todavía ningún registro.

## 3.f Valores fuera de dominio o inconsistentes

In [10]:
variables_categoricas = [
    "DEPARTAMENTO", "NIVEL", "SECTOR", "AREA",
    "STATUS", "MODALIDAD", "JORNADA", "PLAN",
    "DEPARTAMENTAL",
]

for variable in variables_categoricas:
    print(f"\n{variable}")
    display(
        df_raw[variable]
        .fillna("")
        .value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="cantidad")
    )


DEPARTAMENTO


,DEPARTAMENTO,cantidad
0,CIUDAD CAPITAL,2161
1,GUATEMALA,1908
2,SAN MARCOS,724
3,ESCUINTLA,708
4,HUEHUETENANGO,591
5,QUETZALTENANGO,551
6,PETEN,516
7,ALTA VERAPAZ,475
8,SUCHITEPEQUEZ,437
9,CHIMALTENANGO,435



NIVEL


,NIVEL,cantidad
0,DIVERSIFICADO,11867
1,,23



SECTOR


...

,JORNADA,cantidad
0,DOBLE,3772
1,VESPERTINA,3407
2,MATUTINA,3045
3,SIN JORNADA,1099
4,NOCTURNA,415
5,INTERMEDIA,129
6,,23



PLAN


,PLAN,cantidad
0,DIARIO(REGULAR),7452
1,FIN DE SEMANA,2898
2,SEMIPRESENCIAL (FIN DE SEMANA),542
3,SEMIPRESENCIAL (UN DÍA A LA SEMANA),437
4,A DISTANCIA,188
5,SEMIPRESENCIAL,118
6,VIRTUAL A DISTANCIA,70
7,SEMIPRESENCIAL (DOS DÍAS A LA SEMANA),67
8,SABATINO,59
9,DOMINICAL,27



DEPARTAMENTAL


,DEPARTAMENTAL,cantidad
0,GUATEMALA NORTE,1489
1,GUATEMALA OCCIDENTE,1035
2,GUATEMALA SUR,1021
3,SAN MARCOS,724
4,ESCUINTLA,708
5,HUEHUETENANGO,591
6,QUETZALTENANGO,551
7,GUATEMALA ORIENTE,524
8,PETÉN,516
9,ALTA VERAPAZ,475


### Análisis 3.f

> Documente dominios y categorías por revisar.

## 3.g Variables con formatos inconsistentes

In [11]:
diagnostico_formatos = []

for variable in columnas_originales:
    texto = df_raw[variable].fillna("").astype("string")

    diagnostico_formatos.append({
        "variable": variable,
        "espacios_inicio_fin": int(
            texto.ne(texto.str.strip()).sum()
        ),
        "espacios_multiples": int(
            texto.str.contains(r"\s{2,}", regex=True).sum()
        ),
        "punto_final": int(
            texto.str.strip().str.endswith(".").sum()
        ),
    })

tabla_formatos = pd.DataFrame(diagnostico_formatos)
display(
    tabla_formatos.loc[
        tabla_formatos[
            ["espacios_inicio_fin", "espacios_multiples", "punto_final"]
        ].sum(axis=1) > 0
    ]
)

telefono = df_raw["TELEFONO"].fillna("").str.strip()

resumen_telefono = pd.DataFrame({
    "métrica": [
        "Vacíos",
        "Exactamente 8 dígitos",
        "Formato diferente de 8 dígitos",
        "Con letras o texto adicional",
    ],
    "cantidad": [
        telefono.eq("").sum(),
        telefono.str.fullmatch(r"\d{8}").sum(),
        (
            telefono.ne("")
            & ~telefono.str.fullmatch(r"\d{8}")
        ).sum(),
        telefono.str.contains(
            r"[A-Za-zÁÉÍÓÚÑáéíóúñ]",
            regex=True,
        ).sum(),
    ],
})

display(resumen_telefono)

,variable,espacios_inicio_fin,espacios_multiples,punto_final
4,ESTABLECIMIENTO,0,1395,107
5,DIRECCION,0,485,83
6,TELEFONO,0,8,0
7,SUPERVISOR,0,102,21
8,DIRECTOR,0,866,21


,métrica,cantidad
0,Vacíos,969
1,Exactamente 8 dígitos,10670
2,Formato diferente de 8 dígitos,251
3,Con letras o texto adicional,9


## 3.g.1 Detalle de TELEFONO con formato no estándar

In [12]:
import sys
sys.path.append("..")
from src.diagnostico import telefonos_no_estandar

detalle_telefonos = telefonos_no_estandar(df_raw)

print("Total con formato no estándar:", len(detalle_telefonos))
display(detalle_telefonos["categoria"].value_counts())

for categoria in detalle_telefonos["categoria"].unique():
    ejemplos = detalle_telefonos.loc[
        detalle_telefonos["categoria"].eq(categoria), "TELEFONO"
    ].unique()[:5]
    print(f"\n{categoria}: {list(ejemplos)}")

Total con formato no estándar: 251


categoria
varios numeros                           189
numero legado de 7 digitos                35
no interpretable                          21
un numero de 8 digitos mal formateado      3
incluye placeholder de ceros               2
incluye extension                          1
Name: count, dtype: int64


numero legado de 7 digitos: ['4085613', '8392658', '2232068', '2267425', '2223228']

varios numeros: ['78208583-78209143', '79540830-79540909', '78391288-78392217', '79649696-78739432', '78739432-79649696']

no interpretable: ['783928', '230835', '25763,26725 Y 21568', '25763, 26725 Y 21568', '26725,25763 Y 21568']

un numero de 8 digitos mal formateado: ['2220-1723', '2485-4989', '5565-5011']

incluye extension: ['439990 ESTX. 225-250']

incluye placeholder de ceros: ['41724130-00000000']


## 3.g.2 Detalles del establecimiento

In [13]:
import sys
sys.path.append("..")
from src.diagnostico import inventario_escritura, variantes_establecimiento

display(inventario_escritura(df_raw["ESTABLECIMIENTO"]))

variantes = variantes_establecimiento(df_raw)
display(variantes["categoria"].value_counts())
display(variantes[variantes["categoria"].eq("posible duplicado")].head(15))

,patron,celdas,porcentaje,ejemplos
5,tiene tildes,3223,27.11,CENTRO DE FORMACIÓN INTEGRAL CIUDAD DE LA ESPE...
2,comillas,2961,24.90,"COLEGIO ""LA INMACULADA"" ¦ INSTITUTO NORMAL MIX..."
0,espacios multiples,1395,11.73,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN ...
6,abreviatura con punto,584,4.91,INSTITUTO DE TURSMO Y AVIACON DEL NORTE I.T.A....
7,contiene digitos,298,2.51,LICEO GUATEMALTECO 502 ¦ COLEGIO PARTICULAR MI...
3,parentesis,248,2.09,INSTITUTO GUATEMALTECO DE EDUCACION RADIOFONIC...
1,termina en punto,107,0.90,LICEO DE COMPUTACION Y EDUCACION PARA EL HOGAR...
8,caracter inusual,67,0.56,INSTITUTO MIXTO DE EDUCACIÒN BILINGÜE INTERCUL...
4,tiene minusculas,0,0.00,


categoria
posible duplicado    848
homonimos            279
Name: count, dtype: int64

,escrituras,codigos,municipios,registros,formas,categoria
clave,,,,,,
CENTRO DE DESARROLLO EDUCATIVO SAN JOSE,5,5,1,5,"[CENTRO DE DESARROLLO EDUCATIVO SAN JOSE, CEN...",posible duplicado
CENTRO EDUCATIVO JOSE MILLA Y VIDAURRE,5,11,1,11,"[CENTRO EDUCATIVO JOSE MILLA Y VIDAURRE, CENT...",posible duplicado
CENTRO EDUCATIVO PRIVADO MIXTO VISION INTEGRAL CEVI,5,6,1,6,[CENTRO EDUCATIVO PRIVADO MIXTO VISION INTEGR...,posible duplicado
CENTRO TECNOLOGICO INDUSTRIAL CETI,5,5,1,5,"[CENTRO TECNOLOGICO INDUSTRIAL CETI, CENTRO T...",posible duplicado
COLEGIO BILINGUE DE COMPUTACION C B C EL CARMEN,5,5,1,5,[COLEGIO BILINGUE DE COMPUTACION C.B.C EL CAR...,posible duplicado
ESCUELA NORMAL PARA MAESTROS DE EDUCACION MUSICAL JESUS MARIA ALVARADO,5,5,1,5,[ESCUELA NORMAL PARA MAESTROS DE EDUCACIÓN ...,posible duplicado
INSTITUTO PARTICULAR MIXTO DE EDUCACION BASICA Y DIVERSIFICADA DR PEDRO MOLINA,5,5,1,5,[INSTITUTO PARTICULAR MIXTO DE EDUCACION BÁSI...,posible duplicado
INSTITUTO TECNICO PARTICULAR DE DIVERSIFICADO INTECPADI,5,9,1,9,[INSTITUTO TECNICO PARTICULAR DE DIVERSIFICADO...,posible duplicado
CENTRO DE ESTUDIOS TECNICOS Y AVANZADOS DE CHIMALTENANGO NO 2 C E T A CH NO 2,4,5,1,5,[CENTRO DE ESTUDIOS TECNICOS Y AVANZADOS DE CH...,posible duplicado


## 3.g.3 Direcciones

In [14]:
from src.diagnostico import (
    marcadores_direccion, direccion_redundante,
    direccion_contaminada, inventario_escritura,
)

display(marcadores_direccion(df_raw["DIRECCION"]))
display(direccion_redundante(df_raw))
display(direccion_contaminada(df_raw["DIRECCION"]))
display(inventario_escritura(df_raw["DIRECCION"]))

,marcador,celdas,porcentaje,ejemplos
1,via urbana,6914,58.15,6A. AVENIDA 1-15 ZONA 4 ¦ 7A. AVENIDA 11-109 Z...
0,numero de casa (N-N),6178,51.96,6A. AVENIDA 1-15 ZONA 4 ¦ 7A. AVENIDA 11-109 Z...
3,zona,5223,43.93,6A. AVENIDA 1-15 ZONA 4 ¦ KM.2 SALIDA A SAN JU...
5,barrio/colonia,2866,24.10,"DIAGONAL 08 8-05 ZONA 8, BARRIO CANTON LAS CAS..."
4,rural,2595,21.83,"DIAGONAL 08 8-05 ZONA 8, BARRIO CANTON LAS CAS..."
2,kilometro,410,3.45,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8 ¦ KM 20...
6,cabecera/casco,389,3.27,CABECERA ¦ CENTRO URBANO UNO ¦ CASCO URBANO
8,sin ningun marcador,388,3.26,SAN BENITO I ¦ COBAN ¦ TAMAHU
7,solo puntuacion,13,0.11,-- ¦ --- ¦ -----------


,caso,celdas,porcentaje,ejemplos
0,direccion == municipio,169,1.42,COBAN ¦ TAMAHU ¦ SENAHU
1,direccion contiene municipio,948,7.97,7A. CALLE 11-09 ZONA 6 COBAN ¦ DIAGONAL 1 1-26...


,patron,celdas,ejemplos
0,fecha incrustada,26,CASERO CHIJULHA 02/01/2018 ¦ LOTES 682-678 1A....
1,letra O por cero,10,0 CALLE B OC-150 ZONA 4 ¦ 4A. AVENIDA OC-176 Z...


,patron,celdas,porcentaje,ejemplos
7,contiene digitos,8339,70.13,6A. AVENIDA 1-15 ZONA 4 ¦ KM.2 SALIDA A SAN JU...
6,abreviatura con punto,1198,10.08,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8 ¦ 3 AV....
5,tiene tildes,1161,9.76,"DIAGONAL 08 8-05 ZONA 8, BARRIO CANTÓN LAS CAS..."
0,espacios multiples,485,4.08,"12 AV. A 4-30, ZONA 3, BARRIO ASUNCION ¦ 12 A..."
2,comillas,388,3.26,"2A. CALLE ""A"" 11-10, ZONA 1 ¦ 0 CALLE ""C"" 4-29..."
1,termina en punto,83,0.70,"5A. CALLE 1-98, ZONA 3. ¦ O CALLE B OC-150, ..."
8,caracter inusual,67,0.56,"4ª. AVENIDA, ENTRE O Y 1 CALLE ZONA 4, BARRIO ..."
3,parentesis,34,0.29,3A. AVENIDA CINCO GUION DOCE (5-12) ¦ KM. 58 C...
4,tiene minusculas,10,0.08,"3a. CALLE 6-108 ZONA 1, COLONIA LAS MAJADAS ¦ ..."


## 3.g.3 Supervisor y Director

In [15]:
from src.diagnostico import nombres_ausentes, variantes_nombre

for col in ["SUPERVISOR", "DIRECTOR"]:
    print("=" * 64)
    print(col)
    print("=" * 64)
    display(nombres_ausentes(df_raw[col]))

    v = variantes_nombre(df_raw[col], df_raw["DEPARTAMENTAL"])
    print("grupos con >1 escritura del mismo nombre:")
    display(v["categoria"].value_counts())
    display(v.head(8))

SUPERVISOR


,categoria,celdas,porcentaje,ejemplos
0,vacia,558,4.69,
1,solo simbolos,3,0.03,---------------------------- -----------------...
2,un solo token,0,0.00,
3,frase de ausencia,0,0.00,
4,nombre aparente,11329,95.28,JORGE EDUARDO PAQUE LAZARO ¦ JOSE ARTURO CHOC ...


grupos con >1 escritura del mismo nombre:


categoria
misma persona          182
revisar (homonimo?)     10
Name: count, dtype: int64

,escrituras,jurisdicciones,registros,formas,categoria
clave,,,,,
AGRIPINO PEREN CUTZAL,2,1,10,"[AGRIPINO PEREN CUTZAL, AGRIPINO PERÉN CUTZAL]",misma persona
ALBA LUZ CRISTINA AGUSTIN MANUEL,2,1,8,"[ALBA LUZ CRISTINA AGUSTIN MANUEL, ALBA LUZ CR...",misma persona
ALBERTO ANTONIO ORTIZ MONZON,2,1,4,"[ALBERTO ANTONIO ORTIZ MONZON, ALBERTO ANTONIO...",misma persona
AMADO BEATO ESCALANTE GONZALEZ,2,1,7,"[AMADO BEATO ESCALANTE GONZALEZ, AMADO BEATO E...",misma persona
AMANDA CAROLINA LOPEZ MERIDA,2,1,16,"[AMANDA CAROLINA LOPEZ MERIDA, AMANDA CAROLINA...",misma persona
AMARILIS NOHEMI MENDEZ MATEO,2,1,10,"[AMARILIS NOHEMI MENDEZ MATEO, AMARILIS NOHEMI...",misma persona
AMILCAR ROCAEL VELASQUEZ OROZCO,2,1,11,"[AMILCAR ROCAEL VELASQUEZ OROZCO, AMÍLCAR ROCA...",misma persona
ANA LUISA AVILA GOMEZ DE LEMUS,2,1,12,"[ANA LUISA AVILA GÓMEZ DE LEMUS, ANA LUISA ÁVI...",misma persona


DIRECTOR


,categoria,celdas,porcentaje,ejemplos
0,vacia,1755,14.76,
1,solo simbolos,384,3.23,-- ¦ --- ¦ -
2,un solo token,5,0.04,X ¦ XXXXXX ¦ XXX
3,frase de ausencia,30,0.25,SIN DATO ¦ SIN DATOS ¦ SIN DATOS ( TEMPORAL T...
4,nombre aparente,9716,81.72,GUSTAVO ADOLFO SIERRA POP ¦ GILMA DOLORES GUAY...


grupos con >1 escritura del mismo nombre:


categoria
misma persona          170
revisar (homonimo?)     15
Name: count, dtype: int64

,escrituras,jurisdicciones,registros,formas,categoria
clave,,,,,
SARA MARIBEL CHINCHILLA GAITAN,4,1,8,"[SARA MARIBEL CHINCHILLA GAITÁN, SARA ...",misma persona
ANAHI DEL PILAR HERNANDEZ CHOCHOM,3,1,4,"[ANAHI DEL PILAR HERNÁNDEZ CHOCHÓM, ANAHÍ DEL ...",misma persona
ARACELI SANDOVAL VASQUEZ,3,1,4,"[ARACELI SANDOVAL VÁSQUEZ, ARACELI SANDOVAL V...",misma persona
JAVIER SANCHEZ,3,1,8,"[JAVIER SANCHEZ, JAVIER SANCHEZ, JAVIER SÁNCHEZ]",misma persona
JUAN CARLOS MONZON LUTIN,3,1,4,"[JUAN CARLOS MONZON LUTIN, JUAN CARLOS MONZON...",misma persona
MARIA DEL ROSARIO LOPEZ ESCOBAR,3,1,6,"[----MARIA DEL ROSARIO LOPEZ ESCOBAR, MARIA DE...",misma persona
MARIA VERONICA AUYON OCAMPO DE GALICIA,3,1,5,"[MARIA VERONICA AUYON OCAMPO DE GALICIA, MARIA...",misma persona
MARIO GONZALEZ,3,1,4,"[MARIO GONZALEZ, MARIO GONZALEZ, MARIO GONZÁLEZ]",misma persona


## 3.h Problemas potenciales de calidad

In [17]:
# =====================================================================
# 3.h  Codebook de defectos -> problemas_calidad_preliminares.csv
# ---------------------------------------------------------------------
# Tabla COMPARTIDA por los tres turnos. Cada quien llena SU bloque en la
# lista `defectos`. Una fila por defecto (una variable puede tener varias).
#
# Formato de cada fila (tupla):
#   (variable, problema, categoria, cantidad, requiere_limpieza,
#    estrategia_sugerida, ejemplo)
#   - categoria: "faltante" | "formato" | "duplicado" | "contaminacion" | "estructural"
#   - cantidad : n de celdas afectadas (o de grupos/escrituras en duplicados)
#   - requiere_limpieza: "Si" | "No"   (registrar el "No" tambien es diagnostico)
#
# OJO: no sumar la columna `cantidad` entre filas; se solapan por diseno
#      (p.ej. en DIRECTOR los disfrazados son subconjunto de la ausencia).
# =====================================================================

N = len(df_raw)

defectos = []

# ---------------------------------------------------------------------
# TURNO 1 - VIANKA - CODIGO, DISTRITO, DEPARTAMENTO, MUNICIPIO, DEPARTAMENTAL
# Descomenta y completa con tu diagnostico. Ejemplo del formato:
# defectos += [
#     ("DISTRITO", "Descripcion del defecto", "faltante", 555, "Si",
#      "Regla sugerida en una linea", "ejemplo real"),
# ]
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# TURNO 2 - RICARDO - ESTABLECIMIENTO, DIRECCION, TELEFONO, SUPERVISOR, DIRECTOR
# ---------------------------------------------------------------------
defectos += [
    ("TELEFONO", "Vacio", "faltante", 969, "No",
     "Faltante legitimo, dejar NA", ""),
    ("TELEFONO", "Formato no estandar (varios num., legado 7 dig, extension)", "formato", 251, "Si",
     "Separar con separar_numeros() a lista de 8 digitos", "78208583-78209143 | 4085613"),

    ("ESTABLECIMIENTO", "Tildes inconsistentes", "formato", 3223, "Si",
     "Normalizar solo para agrupar; conservar el nombre real", "EDUCACION con y sin tilde"),
    ("ESTABLECIMIENTO", "Comillas decorativas", "formato", 2961, "Si",
     "Quitar comillas (adorno, no sintaxis)", 'COLEGIO "LA INMACULADA"'),
    ("ESTABLECIMIENTO", "Espacios multiples", "formato", 1395, "Si",
     "Colapsar a un solo espacio", "INSTITUTO MIXTO  NOCTURNO"),
    ("ESTABLECIMIENTO", "Variantes de escritura (848 grupos posible dup.)", "duplicado", 1483, "Si",
     "Comparar JORNADA/PLAN/DIRECCION antes de fusionar; CODIGO es la llave", "CENTRO ... SAN JOSE (5 escrituras)"),

    ("DIRECCION", "Vacia", "faltante", 99, "No",
     "Faltante real, dejar NA", ". | ---"),
    ("DIRECCION", "Solo el municipio (faltante disfrazado)", "faltante", 169, "Si",
     "Reclasificar a NA; es un piso, no un total", "COBAN | CHISEC"),
    ("DIRECCION", "Municipio redundante al final", "formato", 948, "Si",
     "Recortar el sufijo del municipio", "... ZONA 5 HUEHUETENANGO"),
    ("DIRECCION", "Fecha incrustada al final", "contaminacion", 26, "Si",
     "Recortar la fecha final (exige dos separadores)", "... 22/02/20"),
    ("DIRECCION", "Letra O por cero", "contaminacion", 10, "Si",
     "Corregir O->0 en contexto numerico", "O AV. 0-06 | OC-150"),

    ("SUPERVISOR", "Ausencia real", "faltante", 561, "Si",
     "Imputable: por DISTRITO en el plan", ""),
    ("SUPERVISOR", "Variantes por tildes del mismo nombre", "duplicado", 192, "Si",
     "Fusionar con normalizar_nombre()", "GONZALEZ DE LEON con y sin tilde"),
    ("SUPERVISOR", "Caracter raro (O/0, tilde grave, apostrofo)", "contaminacion", 16, "Si",
     "Corregir grafias en contexto", "ACEVED0 | ORTIZ grave | O'NELL"),

    ("DIRECTOR", "Ausencia real (incluye 360 disfrazados que 3.c no vio)", "faltante", 2174, "Si",
     "Convertir disfrazados a NA primero; no imputable", "--- | 000000 | SIN DATOS"),
    ("DIRECTOR", "Variantes por tildes del mismo nombre", "duplicado", 198, "Si",
     "Fusionar con normalizar_nombre()", "VASQUEZ vs VASQUEZ REYES"),
    ("DIRECTOR", "Titulo incrustado (LIC., PEM.)", "formato", 18, "Si",
     "Separar el titulo del nombre", "LIC. ELGI WALTER BOTEO GARCIA"),
]

# ---------------------------------------------------------------------
# TURNO 3 - NADISSA - NIVEL, SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN
# Descomenta y completa con tu diagnostico. Ejemplo del formato:
# defectos += [
#     ("AREA", "Descripcion del defecto", "formato", 3, "Si",
#      "Regla sugerida en una linea", "SIN ESPECIFICAR"),
# ]
# ---------------------------------------------------------------------

problemas = pd.DataFrame(defectos, columns=[
    "variable", "problema", "categoria", "cantidad",
    "requiere_limpieza", "estrategia_sugerida", "ejemplo",
])
problemas["porcentaje"] = (problemas["cantidad"] / N * 100).round(2)
problemas = problemas[[
    "variable", "problema", "categoria", "cantidad", "porcentaje",
    "requiere_limpieza", "estrategia_sugerida", "ejemplo",
]]

problemas.to_csv(
    REPORT_DIR / "problemas_calidad_preliminares.csv",
    index=False, encoding="utf-8-sig",
)
print("Defectos registrados:", len(problemas))
print("Variables con limpieza requerida:",
      problemas.loc[problemas["requiere_limpieza"].eq("Si"), "variable"].nunique())
display(problemas)


Defectos registrados: 17
Variables con limpieza requerida: 5


,variable,problema,categoria,cantidad,porcentaje,requiere_limpieza,estrategia_sugerida,ejemplo
0,TELEFONO,Vacio,faltante,969,8.15,No,"Faltante legitimo, dejar NA",
1,TELEFONO,"Formato no estandar (varios num., legado 7 dig...",formato,251,2.11,Si,Separar con separar_numeros() a lista de 8 dig...,78208583-78209143 | 4085613
2,ESTABLECIMIENTO,Tildes inconsistentes,formato,3223,27.11,Si,Normalizar solo para agrupar; conservar el nom...,EDUCACION con y sin tilde
3,ESTABLECIMIENTO,Comillas decorativas,formato,2961,24.90,Si,"Quitar comillas (adorno, no sintaxis)","COLEGIO ""LA INMACULADA"""
4,ESTABLECIMIENTO,Espacios multiples,formato,1395,11.73,Si,Colapsar a un solo espacio,INSTITUTO MIXTO NOCTURNO
5,ESTABLECIMIENTO,Variantes de escritura (848 grupos posible dup.),duplicado,1483,12.47,Si,Comparar JORNADA/PLAN/DIRECCION antes de fusio...,CENTRO ... SAN JOSE (5 escrituras)
6,DIRECCION,Vacia,faltante,99,0.83,No,"Faltante real, dejar NA",. | ---
7,DIRECCION,Solo el municipio (faltante disfrazado),faltante,169,1.42,Si,"Reclasificar a NA; es un piso, no un total",COBAN | CHISEC
8,DIRECCION,Municipio redundante al final,formato,948,7.97,Si,Recortar el sufijo del municipio,... ZONA 5 HUEHUETENANGO
9,DIRECCION,Fecha incrustada al final,contaminacion,26,0.22,Si,Recortar la fecha final (exige dos separadores),... 22/02/20


### Análisis 3.h — Problemas de calidad y estrategia preliminar (TURNO 2)

Los defectos de las cinco variables caen en cuatro categorías: **faltantes** (incluidos los disfrazados), **formato**, **duplicados** y **contaminación**. El detalle cuantificado está en `problemas_calidad_preliminares.csv`; acá va la lectura.

**1. Los conteos de nulos son un piso, no un total.** La primera regla del plan es normalizar a NA los faltantes disfrazados *antes* de medir: en DIRECTOR son 360 celdas (`---`, `000000`, `X`, `SIN DATOS…`) que suben la ausencia real de 15.3 % a **18.3 %**; en DIRECCION son 169 que solo repiten el municipio. Ninguno lo ve un conteo de vacíos.

**2. El faltante se trata según por qué falta.** TELEFONO (8.1 %) y DIRECTOR (18.3 %) faltan de forma legítima y no se deducen de otra columna: se dejan NA. SUPERVISOR (4.7 %) es recuperable —un distrito tiene un supervisor asignado—, así que se marca para imputar por distrito en el plan.

**3. Los duplicados no se eliminan por nombre.** ESTABLECIMIENTO no identifica una entidad (cientos de institutos comparten nombre en municipios distintos); la llave es CODIGO. Antes de fusionar hay que comparar JORNADA, PLAN y DIRECCION dentro de cada grupo. En SUPERVISOR y DIRECTOR, en cambio, fusionar variantes por tildes es más seguro porque cada nombre es una persona real atada a una jurisdicción.

**4. La contaminación es poca pero invisible.** Fechas incrustadas (26), letra O por cero (10) y títulos como LIC./PEM. (18) producen cadenas perfectamente válidas: ningún filtro sintáctico las atrapa, hay que buscarlas a propósito.

**Lección de diseño transversal:** la puntuación es *sintaxis* en unas columnas y *adorno* en otras. En un nombre la comilla y el guion se pueden borrar; en una dirección delimitan la vía y el número de casa. La regla de limpieza no puede ser global.

Esto es diagnóstico. El **plan de limpieza** (`docs/plan_limpieza.md`) formaliza el orden y el *cómo*; aquí las reglas quedan solo como propuesta.
